In [189]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle

In [190]:
data = pd.read_csv("Churn_Modelling.csv")
data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)
label_encoder_gender = LabelEncoder()
data["Gender"] = label_encoder_gender.fit_transform(data["Gender"])

In [191]:
#from sklearn.preprocessing import OneHotEncoder
onehot_encoder_geo = OneHotEncoder() #sparse_output=False
geo_encoder = onehot_encoder_geo.fit_transform(data[["Geography"]]) 
geo_encoded_df = pd.DataFrame(geo_encoder.toarray(),columns=onehot_encoder_geo.get_feature_names_out(["Geography"]))
data =pd.concat([data.drop("Geography",axis=1),geo_encoded_df],axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [192]:
#Split Data into Feature and Target Feature.
X = data.drop("EstimatedSalary",axis=1)
Y = data["EstimatedSalary"]

In [193]:
#split the data in trainning and testing sets.
X_train, X_test, Y_train, Y_test = train_test_split(X,Y,test_size=0.2,random_state= 42)
#Scale these feature
scaler = StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [194]:
#split the data in trainning and testing sets.
X_train, X_test, Y_train, Y_test = train_test_split(X,Y,test_size=0.2,random_state= 42)


In [195]:

with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)

with open('onehot_encoder_geo.pkl','wb') as file:
    pickle.dump(onehot_encoder_geo,file)
    
with open("scaler.pkl","wb") as file:
    pickle.dump(scaler,file)

In [196]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime



In [197]:
model = Sequential([
  Dense(64,activation="relu",input_shape=(X_train.shape[1],)), #First Hidden Layer connected with Input later
  Dense(32,activation='relu'), #HL2
  Dense(1)#output layer
])



##Complie the model.
model.compile(optimizer="adam",loss="mean_absolute_error",metrics=["mae"])

model.summary()

Model: "sequential_15"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_45 (Dense)            (None, 64)                832       
                                                                 
 dense_46 (Dense)            (None, 32)                2080      
                                                                 
 dense_47 (Dense)            (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [198]:
log_dir = "regression/fit/"+datetime.datetime.now().strftime("%Y%m%d-%H%m%S")
tensorflow_callaback = TensorBoard(log_dir=log_dir,histogram_freq=1)

In [199]:
#set up Early Stopping

early_stopping_callback=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [200]:
#Training the model
history = model.fit(X_train,Y_train,
                    validation_data=(X_test,Y_test),
                    epochs=100,
                    callbacks=[tensorflow_callaback,early_stopping_callback]
                    )

Epoch 1/100
250/250 [==============================] - 1s 2ms/step - loss: 71178.6016 - mae: 71178.6016 - val_loss: 66146.6328 - val_mae: 66146.6328
Epoch 2/100
250/250 [==============================] - 0s 2ms/step - loss: 63775.3320 - mae: 63775.3320 - val_loss: 58503.5195 - val_mae: 58503.5195
Epoch 3/100
250/250 [==============================] - 0s 1ms/step - loss: 55042.6016 - mae: 55042.6016 - val_loss: 51527.9258 - val_mae: 51527.9258
Epoch 4/100
250/250 [==============================] - 0s 2ms/step - loss: 52041.5703 - mae: 52041.5703 - val_loss: 50980.7852 - val_mae: 50980.7852
Epoch 5/100
250/250 [==============================] - 0s 2ms/step - loss: 51765.4766 - mae: 51765.4766 - val_loss: 51214.7148 - val_mae: 51214.7148
Epoch 6/100
250/250 [==============================] - 0s 1ms/step - loss: 51511.0977 - mae: 51511.0977 - val_loss: 50986.9961 - val_mae: 50986.9961
Epoch 7/100
250/250 [==============================] - 0s 2ms/step - loss: 51282.9102 - mae: 51282.9102 - 

In [201]:
#Load Tenserboard Extension
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [202]:
#%tensorboard --logdir regression/fit

In [203]:
test_loss,test_mae = model.evaluate(X_test,Y_test)
print(f'Test MAE : {test_mae}')

63/63 [==============================] - 0s 935us/step - loss: 50312.5352 - mae: 50312.5352
Test MAE : 50312.53515625


In [204]:
model.save("regression_mode.h5")

/home/viral/projects/genai-lab/ann-demo/venv/lib/python3.11/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
